# DuckDB documentation — embedded SQL, from the basics

Living reference for every DuckDB command and idiom this repo uses. Each section:
what the command does, then a runnable example on our actual data.

**The architecture in one paragraph:** DuckDB is an *embedded* database — the whole SQL
engine is a library inside this Python process (SQLite's analytical sibling: columnar,
vectorized). `duckdb.connect()` returns a library handle, **not** a network connection;
there is no server, no port, no credentials. Storage is Parquet files on disk. That makes
the whole project single-tier: app code, engine, and data in one process on one machine.

Data layers this notebook touches:
- `data/staging/statsbomb/{events,lineups,matches}/*.parquet` — flat event tables
- `data/spadl/statsbomb/*.parquet` — the action fact table + 3 dimension tables
- `data/features/player_season_context.parquet` — one row per (player, season, context)
- `data/football.duckdb` — a database *file* containing only views + one small table

In [1]:
from pathlib import Path
import duckdb
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "config.yaml").exists() else Path.cwd().parent
print("duckdb", duckdb.__version__)

duckdb 1.5.5


## 1. Connecting

- `duckdb.connect()` — fresh **in-memory** database. Dies with the process. Use for scratch.
- `duckdb.connect("file.duckdb")` — opens/creates a **database file** (persists tables/views).
- `read_only=True` — required etiquette for shared files: multiple processes may read one
  database file simultaneously, but only one may hold it read-write.
- `con.close()` — releases the handle (in-memory contents are gone after this).

In [2]:
con = duckdb.connect()                                            # in-memory scratch
db = duckdb.connect(str(ROOT / "data/football.duckdb"), read_only=True)   # our view layer
print(db.sql("SHOW TABLES"))

┌───────────────────────┐
│         name          │
│        varchar        │
├───────────────────────┤
│ competitions          │
│ events                │
│ lineups               │
│ matches               │
│ player_match_minutes  │
│ player_season_context │
└───────────────────────┘



## 2. `.sql()` vs `.execute()` — the two ways to run SQL

Both run a statement; they differ in what comes back.

- **`con.sql("...")`** returns a **Relation** — a *lazy* result you can print (it
  pretty-prints a preview), chain further SQL onto, or materialize with `.df()`,
  `.fetchall()`, `.fetchone()`, `.arrow()`. Preferred for exploration.
- **`con.execute("...")`** is the classic **DB-API (PEP 249)** call: it runs eagerly and
  returns the connection itself, so you chain a fetch: `.execute(q).fetchone()`.
  Preferred for parameterized statements and INSERT/COPY-style commands.

**Parameters** (both methods): positional `?` filled from a list, or named `$name` filled
from a dict. Always use parameters for values that vary — never f-string a value into SQL
(f-strings in this repo are used for *paths*, which is fine; values go through `?`).

In [3]:
rel = con.sql("SELECT 6 * 7 AS answer")     # lazy Relation — nothing fetched yet
print(rel)                                  # printing shows a preview table

row = con.execute("SELECT ? * ? AS answer", [6, 7]).fetchone()
print("execute + fetchone:", row)

named = con.execute("SELECT $a * $b AS answer", {"a": 6, "b": 7}).fetchone()
print("named params:", named)

┌────────┐
│ answer │
│ int32  │
├────────┤
│     42 │
└────────┘

execute + fetchone: (42,)
named params: (42,)


## 3. Getting results out

| call | returns | use when |
|---|---|---|
| `.fetchone()` | one tuple (or None) | single value/row checks, assertions |
| `.fetchall()` | list of tuples | small result sets |
| `.df()` | pandas DataFrame | anything you'll plot or manipulate |
| `.arrow()` | Arrow table | zero-copy handoff to other tools |
| `.show()` / `print(rel)` | pretty preview | eyeballing |

The same methods exist on both a Relation (from `.sql`) and the connection (after `.execute`).

In [ ]:
FEATURES = f"{ROOT}/data/features/player_season_context.parquet"

df = con.sql(f"SELECT player, season, context, carry_share_of_actions, left_foot_share_of_foot_actions "
             f"FROM '{FEATURES}' ORDER BY actions DESC LIMIT 5").df()
df

## 4. Querying files directly — the reason DuckDB is here

No import step, no schema declaration: `FROM 'path'` reads Parquet/CSV/JSON in place,
and **globs** query many files as one table. This is how 0.8 GB of Parquet behaves like
a database without ever being loaded.

- `FROM 'dir/*.parquet'` — glob over files (our per-comp-season partitioning)
- `read_csv_auto('x.csv.gz')` — CSV with inferred types (handles gzip transparently)
- `read_json_auto('x.json')` — JSON arrays/objects to rows

In [5]:
SPADL = f"{ROOT}/data/spadl/statsbomb"

print(con.sql(f"SELECT count(*) AS actions, count(DISTINCT game_id) AS games "
              f"FROM '{SPADL}/[0-9]*.parquet'"))

print(con.sql(f"SELECT * FROM read_csv_auto('{ROOT}/data/raw/reep/players.csv.gz') LIMIT 3"))

print(con.sql(f"SELECT competition_name, season_name FROM read_json_auto("
              f"'{ROOT}/data/raw/statsbomb/data/competitions.json') LIMIT 3"))

┌─────────┬───────┐
│ actions │ games │
│  int64  │ int64 │
├─────────┼───────┤
│ 7775282 │  3846 │
└─────────┴───────┘



┌──────────────────┬─────────┬────────────────────────────────┬─────────┬─────────┐
│     reep_id      │ status  │             label              │ gender  │ country │
│     varchar      │ varchar │            varchar             │ varchar │ varchar │
├──────────────────┼─────────┼────────────────────────────────┼─────────┼─────────┤
│ rp0000083e5c0fcb │ active  │ Hugo Mallo Novegil             │ NULL    │ NULL    │
│ rp0000a453dc012b │ active  │ Diego Alejandro Puentes Chávez │ men     │ NULL    │
│ rp000102170da478 │ active  │ Marco Condemi                  │ men     │ NULL    │
└──────────────────┴─────────┴────────────────────────────────┴─────────┴─────────┘

┌────────────────────────┬─────────────┐
│    competition_name    │ season_name │
│        varchar         │   varchar   │
├────────────────────────┼─────────────┤
│ 1. Bundesliga          │ 2023/2024   │
│ 1. Bundesliga          │ 2015/2016   │
│ African Cup of Nations │ 2023        │
└────────────────────────┴─────────────┘

## 5. Views, tables, and `register` — three ways to name data

- `CREATE VIEW v AS SELECT ...` — saves the *query*, not the data. Reads go back to the
  Parquet every time. Our `football.duckdb` is almost entirely views — 524 KB for a
  "database" over 800 MB of Parquet.
- `CREATE TABLE t AS SELECT ...` — materializes rows *into the .duckdb file* (we do this
  only for the tiny `competitions` lookup).
- `con.register("name", df)` — exposes an in-memory **pandas DataFrame as a virtual
  table** for SQL. Used in `identity_join.py` and the notebooks to join Python-side data
  against Parquet without writing it anywhere.

In [6]:
lookup = pd.DataFrame({"pos_group": ["GK", "DF", "MF", "FW"],
                       "line": ["keeper", "back", "middle", "front"]})
con.register("lookup", lookup)

print(con.sql(f"""
    SELECT l.line, count(*) AS entities
    FROM '{FEATURES}' f JOIN lookup l USING (pos_group)
    GROUP BY ALL ORDER BY entities DESC"""))

┌─────────┬──────────┐
│  line   │ entities │
│ varchar │  int64   │
├─────────┼──────────┤
│ back    │     5968 │
│ middle  │     5956 │
│ front   │     4626 │
│ keeper  │     1276 │
└─────────┴──────────┘



## 6. Exploration helpers

- `DESCRIBE SELECT ...` — column names + types of any query without running it fully.
- `SUMMARIZE tbl` — min/max/nulls/approx-uniques per column, one command. First thing to
  run on unfamiliar data.

In [ ]:
print(con.sql(f"DESCRIBE SELECT * FROM '{FEATURES}'").df().head(8).to_string(index=False))
print(con.sql(f"SUMMARIZE (SELECT pass_share_of_actions, carry_share_of_actions, minutes FROM '{FEATURES}')")
      .df()[["column_name", "min", "max", "avg", "null_percentage"]].to_string(index=False))

### Schema + first rows, table by table

Everything below runs through the `duckdb` Python module — the engine is *in this process*
(no psycopg2, no server, no socket; psycopg2 is a network client for a Postgres server,
which this project doesn't have). `con.sql()` → `.df()` is the whole stack.


In [8]:
# every view in football.duckdb: schema + first rows
from IPython.display import display

pd.set_option("display.max_columns", None)   # show every column of the head

for (t,) in db.sql("SHOW TABLES").fetchall():
    n = db.sql(f"SELECT count(*) FROM {t}").fetchone()[0]
    cols = db.sql(f"DESCRIBE {t}").df()
    print(f"\n\u2500\u2500 {t} \u2014 {n:,} rows \u00d7 {len(cols)} cols \u2500\u2500")
    print(cols[["column_name", "column_type"]].to_string(index=False))
    display(db.sql(f"SELECT * FROM {t} LIMIT 3").df())



── competitions — 80 rows × 6 cols ──
   column_name column_type
competition_id     INTEGER
     season_id     INTEGER
   competition     VARCHAR
        season     VARCHAR
        gender     VARCHAR
    is_country     BOOLEAN


,competition_id,season_id,competition,season,gender,is_country
0,9,281,1. Bundesliga,2023/2024,male,False
1,9,27,1. Bundesliga,2015/2016,male,False
2,1267,107,African Cup of Nations,2023,male,True



── events — 13,567,319 rows × 55 cols ──
       column_name column_type
          match_id      BIGINT
    competition_id     INTEGER
         season_id     INTEGER
          event_id     VARCHAR
             index     INTEGER
            period     INTEGER
         timestamp     VARCHAR
            minute     INTEGER
            second     INTEGER
              type     VARCHAR
           team_id     INTEGER
              team     VARCHAR
         player_id      BIGINT
            player     VARCHAR
          position     VARCHAR
                 x       FLOAT
                 y       FLOAT
          duration       FLOAT
        possession     INTEGER
possession_team_id     INTEGER
      play_pattern     VARCHAR
    under_pressure     BOOLEAN
      counterpress     BOOLEAN
        off_camera     BOOLEAN
               out     BOOLEAN
           outcome     VARCHAR
         body_part     VARCHAR
         technique     VARCHAR
    related_events   VARCHAR[]
        pass_end_x       FLO

,match_id,competition_id,season_id,event_id,index,period,timestamp,minute,second,type,team_id,team,player_id,player,position,x,y,duration,possession,possession_team_id,play_pattern,under_pressure,counterpress,off_camera,out,outcome,body_part,technique,related_events,pass_end_x,pass_end_y,pass_length,pass_angle,pass_height,pass_recipient_id,pass_type,pass_cross,pass_switch,pass_through_ball,pass_cut_back,pass_shot_assist,pass_goal_assist,carry_end_x,carry_end_y,dribble_nutmeg,dribble_overrun,dribble_no_touch,shot_end_x,shot_end_y,shot_end_z,shot_xg,shot_type,shot_first_time,shot_freeze_frame,tactics
0,3750234,116,68,d0ba3e2a-4d4a-46c3-9ff4-b6b995414d71,1,1,00:00:00.000,0,0,Starting XI,2735,NY Cosmos,<NA>,None,None,NaN,NaN,0.0,1,2735,Regular Play,False,False,False,False,None,None,None,<NA>,NaN,NaN,NaN,NaN,None,<NA>,None,False,False,False,False,False,False,NaN,NaN,False,False,False,NaN,NaN,NaN,NaN,None,False,None,"{""formation"": 4231, ""lineup"": [{""player"": {""id..."
1,3750234,116,68,1bcef0fc-f75e-49a0-9dff-2a2b31684218,2,1,00:00:00.000,0,0,Starting XI,488,Seattle Sounders,<NA>,None,None,NaN,NaN,0.0,1,2735,Regular Play,False,False,False,False,None,None,None,<NA>,NaN,NaN,NaN,NaN,None,<NA>,None,False,False,False,False,False,False,NaN,NaN,False,False,False,NaN,NaN,NaN,NaN,None,False,None,"{""formation"": 4231, ""lineup"": [{""player"": {""id..."
2,3750234,116,68,2033471c-8c30-474c-a219-3ce84c954016,3,1,00:00:00.000,0,0,Half Start,2735,NY Cosmos,<NA>,None,None,NaN,NaN,0.0,1,2735,Regular Play,False,False,False,False,None,None,None,[d91f72fc-6857-4d03-a72f-00db28374f61],NaN,NaN,NaN,NaN,None,<NA>,None,False,False,False,False,False,False,NaN,NaN,False,False,False,NaN,NaN,NaN,NaN,None,False,None,None



── lineups — 188,410 rows × 17 cols ──
   column_name column_type
      match_id      BIGINT
competition_id     INTEGER
     season_id     INTEGER
       team_id     INTEGER
          team     VARCHAR
     player_id      BIGINT
        player     VARCHAR
      nickname     VARCHAR
 jersey_number     INTEGER
       country     VARCHAR
      position     VARCHAR
     from_time     VARCHAR
       to_time     VARCHAR
   from_period     INTEGER
     to_period     INTEGER
  start_reason     VARCHAR
    end_reason     VARCHAR


...

,match_id,competition_id,season_id,competition,season,match_date,kick_off,match_week,competition_stage,stadium,home_team_id,home_team,home_score,away_team_id,away_team,away_score,has_360
0,3750234,116,68,North American League,1977,1977-08-28,12:00:00.000,1,Final,Providence Park,2735,NY Cosmos,2,488,Seattle Sounders,1,False
1,9880,11,1,La Liga,2017/2018,2018-04-14,16:15:00.000,32,Regular Season,Spotify Camp Nou,217,Barcelona,2,207,Valencia,1,False
2,9912,11,1,La Liga,2017/2018,2018-04-29,20:45:00.000,35,Regular Season,Estadio Abanca-Riazor,219,RC Deportivo La Coruña,2,217,Barcelona,4,False



── player_match_minutes — 110,142 rows × 6 cols ──
column_name column_type
   match_id      BIGINT
  player_id      BIGINT
     player     VARCHAR
    team_id     INTEGER
       team     VARCHAR
    minutes      DOUBLE


,match_id,player_id,player,team_id,team,minutes
0,22945,25718,Chiamaka Cynthia Nnadozie,1213,Nigeria Women's,94.0
1,3879830,25430,Felipe Melo de Carvalho,238,Inter Milan,93.0
2,3825583,26633,Alberto Lora Ramos,1041,Sporting Gijón,95.0



── player_season_context — 17,666 rows × 7 cols ──
 column_name column_type
   player_id      BIGINT
      player     VARCHAR
      season     VARCHAR
     context     VARCHAR
competitions     VARCHAR
     matches      BIGINT
     actions      BIGINT


,player_id,player,season,context,competitions,matches,actions
0,39705,Giorgio Chinaglia,1977,club,North American League,1,150
1,39710,Terry Graham Garbett,1977,club,North American League,1,91
2,4406,Aymen Abdennour,2015/2016,club,La Liga,22,2794


In [ ]:
# same tour for the parquet layers that are not registered as views
LAYERS = {
    "spadl actions":            f"{SPADL}/[0-9]*.parquet",
    "spadl actiontypes":        f"{SPADL}/actiontypes.parquet",
    "spadl results":            f"{SPADL}/results.parquet",
    "spadl bodyparts":          f"{SPADL}/bodyparts.parquet",
    "identity map":             f"{ROOT}/data/identity/player_map.parquet",
    "features (raw)":           FEATURES,
    "features (team-adjusted)": f"{ROOT}/data/features/player_season_team_adjusted.parquet",
}

for name, path in LAYERS.items():
    n = con.sql(f"SELECT count(*) FROM '{path}'").fetchone()[0]
    cols = con.sql(f"DESCRIBE SELECT * FROM '{path}'").df()
    print(f"\n\u2500\u2500 {name} \u2014 {n:,} rows \u00d7 {len(cols)} cols \u2500\u2500")
    print(cols[["column_name", "column_type"]].to_string(index=False))
    display(con.sql(f"SELECT * FROM '{path}' LIMIT 3").df())


## 7. Idioms used in this repo — the reference

| idiom | what it does | used in |
|---|---|---|
| `GROUP BY ALL` | group by every non-aggregate column in the SELECT — no list drift | every script |
| `count(*) FILTER (cond)` | conditional aggregation without CASE spam | build_features.py (action mix) |
| `JOIN ... USING (a, b)` | equi-join on same-named cols, col appears once | everywhere |
| `SELECT x.* EXCLUDE (c)` | all columns except some | build_features.py final join |
| `PIVOT ... ON col USING agg` | long → wide (zone shares become 30 columns) | build_features.py |
| `row_number() OVER (PARTITION BY ...)` | pick "the top one per group" | primary position |
| `sum(x) OVER (PARTITION BY ...)` | group total on each row (shares!) | zone shares |
| `string_agg(DISTINCT x, ' + ')` | collapse group values into one string | competitions per entity |
| `COPY (SELECT ...) TO 'f.parquet' (FORMAT parquet, COMPRESSION zstd)` | write results as Parquet | features, identity |
| `coalesce(a, b)` / `NULLIF(d, 0)` | fallbacks / divide-by-zero guards | identity, minutes |
| `lag(x) OVER (...)` | previous row's value (monotonicity checks) | validate.py |

Runnable mini-examples:

In [ ]:
# FILTER + GROUP BY ALL + USING + window share, in one small query:
# per position group, share of entities that carry more than they pass
print(con.sql(f"""
    SELECT pos_group,
           count(*) AS entities,
           count(*) FILTER (carry_share_of_actions > pass_share_of_actions) / count(*)::DOUBLE AS carry_dominant_share
    FROM '{FEATURES}'
    WHERE pos_group IS NOT NULL
    GROUP BY ALL ORDER BY carry_dominant_share DESC"""))

# row_number to pick one row per group: each group's biggest carrier
print(con.sql(f"""
    SELECT pos_group, player, season, carry_share_of_actions FROM (
        SELECT *, row_number() OVER (PARTITION BY pos_group ORDER BY carry_share_of_actions DESC) AS rn
        FROM '{FEATURES}' WHERE actions >= 1000)
    WHERE rn = 1"""))

In [11]:
# PIVOT: long -> wide (the trick behind the 30 zone-share columns)
print(con.sql(f"""
    PIVOT (SELECT pos_group, context, count(*) AS n
           FROM '{FEATURES}' WHERE pos_group IS NOT NULL GROUP BY ALL)
    ON context USING sum(n)"""))

# COPY ... TO: how every output parquet in this repo is written
con.sql(f"COPY (SELECT player, carry_share_of_actions FROM '{FEATURES}' LIMIT 100) "
        "TO '/tmp/demo_copy.parquet' (FORMAT parquet, COMPRESSION zstd)")
print(con.sql("SELECT count(*) FROM '/tmp/demo_copy.parquet'"))

┌───────────┬────────┬─────────┐
│ pos_group │  club  │ country │
│  varchar  │ int128 │ int128  │
├───────────┼────────┼─────────┤
│ FW        │   3294 │    1332 │
│ DF        │   4219 │    1749 │
│ GK        │    954 │     322 │
│ MF        │   4244 │    1712 │
└───────────┴────────┴─────────┘

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│          100 │
└──────────────┘



## 8. Gotchas this repo actually hit (learn from our bruises)

1. **Window functions + `GROUP BY ALL` don't mix** in one SELECT — "Cannot mix aggregates
   with non-aggregated columns". Split into two CTEs: aggregate first, window second
   (see `zone_counts` → `zones` in build_features.py).
2. **`USING` becomes ambiguous** if *both* sides carry the join columns redundantly —
   drop the unneeded join (our lineups already carried competition_id/season_id).
3. **Empty CSVs infer VARCHAR** — an empty overrides file made `coalesce(varchar, bigint)`
   explode. Cast explicitly when a file may be empty (identity_join.py).
4. **Timestamps as strings** compare correctly *only* because StatsBomb zero-pads
   HH:MM:SS.mmm — lexicographic = chronological. Don't rely on this for general data.
5. **`read_only=True`** whenever another process might hold the .duckdb file — a
   read-write handle takes an exclusive lock.
6. **f-strings are for paths, `?` params are for values.** Path interpolation is safe and
   necessary; value interpolation is how you get injection bugs and quoting hell.

In [12]:
con.close()
db.close()
print("handles released — nothing to shut down, because nothing was running")

handles released — nothing to shut down, because nothing was running
